[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/developability/developability.ipynb)

# Developable Surfaces

This example flows the curved Stanford bunny surface toward a piecewise developable surface following Stein, Grinspun, and Crane (2018). A developable surface unrolls onto a plane without stretching or tearing, like a bent sheet of paper, because it curves in at most one direction at each point. Flowing a doubly curved surface down a developability energy flattens its rounded flanks into smooth ruled patches separated by sharp creases.

Vertices, edge displacements, and ruling directions are intrinsic points, so joining two edges of a triangle with `&` gives its oriented normal patch `edges[0] & edges[1]` with norm equal to twice the triangle area. Around each vertex, the angle-weighted sum of normal dyads `normals.dual() * (normals & Point)` forms a symmetric map from directions to directions whose smallest eigenvalue vanishes whenever a single ruling direction meets every incident face normal at a right angle. Differentiating that minimal quadratic form produces an extrinsic gradient covector that pairs with a vertex displacement via `gradient & step`, and `.dual()` converts the stiffness-scaled gradient into the vertex step.

In matrix geometry-processing notation, each triangle $(i, j, k)$ requires a cross-product unit normal $\mathbf n_{ijk} = (\mathbf x_j - \mathbf x_i) \times (\mathbf x_k - \mathbf x_i) / \|(\mathbf x_j - \mathbf x_i) \times (\mathbf x_k - \mathbf x_i)\|$ and $3 \times 3$ outer products assembled into the vertex covariance matrix $C_i = \sum_{ijk \ni i} \theta_i^{jk}\,\mathbf n_{ijk}\mathbf n_{ijk}^\mathsf{T}$, whose smallest eigenvalue $E = \sum_i \lambda_{\min}(C_i)$ is differentiated through $3 \times 3$ cross-product normal Jacobians $(I - \mathbf n\mathbf n^\mathsf{T})[\mathbf e]_\times / (2A)$ and cotangent corner-angle gradients.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax

from numga import Extensor, NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.surfaces.developability import render
from examples.surfaces.developability.scenarios import bunny_surface

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Patch = ga.gatype.antibivector()                    # Antibivector-

## Normal spread around each vertex

A smooth surface is developable wherever the normals in a neighbourhood of each point lie on a single great circle of the unit sphere, turning around one ruling direction. At a vertex of a triangulated surface, the surrounding triangle normals lie on one great circle whenever a single direction meets all of them at a right angle; the angle-weighted sum of dyads `normals.dual() * (normals & Point)` measures how much the normals lean along any trial direction, so its smallest eigenvalue vanishes on a developable patch and on a sharp crease between two patches alike.

Weighting each triangle's dyad by its interior corner angle makes the spread depend on the surface geometry rather than on how flat regions are triangulated, since splitting a triangle into two preserves the total angle at the shared vertex.

In [ ]:
def corner_angles(topology: Topology, edges: Point) -> Scalar:
    """Interior angles (`[3] Scalar+[F]`) at the three corners of each triangle measure the turn between the two incident `edges` (`[3] Point-[F]`)."""
    tails, heads = topology.corners[2, 1]                                       # [3] Scalar+[F <- F], [3] Scalar+[F <- F]
    next_edges, previous_edges = edges[tails], edges[heads]                     # [3] Point-[F], [3] Point-[F]
    return (-(next_edges.dual_inverse() & previous_edges) / (next_edges.norm() * previous_edges.norm())).arccos()  # [3] Scalar+[F]


def normal_spread(topology: Topology, edges: Point, angles: Scalar) -> tuple[Patch, Extensor]:
    """Unit triangle patches (`Patch-[F]`) and angle-weighted normal-dyad maps (`(Point- <- Point-)[V]`) measure how far the faces around each vertex spread from a single great circle."""
    # Two edge displacements meet in the triangle's oriented normal patch.
    normals = (edges[0] & edges[1]).normalized()                                # Patch-[F]
    incident_angles = (topology.incidence[2, 0].adjugate() * spdiag(angles)).sum(axis=0)  # Scalar+[V <- F]
    return normals, incident_angles * (normals.dual() * (normals & Point))      # Patch-[F], (Point- <- Point-)[V]


topology, vertices = bunny_surface()                                            # Point-[V]
edges = topology.boundary[2, 1] * (topology.d[0] * vertices)                    # [3] Point-[F]
angles = corner_angles(topology, edges)                                         # [3] Scalar+[F]
normals, spreads = normal_spread(topology, edges, angles)                       # Patch-[F], (Point- <- Point-)[V]
eigenvalues, directions = spreads.batch().eigh()                                # [V, 3] Scalar+, [V, 3] Point-
print("initial developability energy:", eigenvalues[:, 0].sum(axis=0).to_array())
save_figure(render.surface_figure(topology, vertices), "developability_bunny");

Flat per-triangle shading lights each triangle by its own normal patch so smooth doubly curved flanks appear softly faceted before the flow sharpens them into planar panels and creases.

## Analytic energy gradient

Because the smallest eigenvalue minimizes the normal-spread quadratic form over unit directions, its first-order change equals the change of that form at fixed minimizing direction as each triangle's normal patch turns and as its three corner angles open or close.

In [ ]:
def energy_gradient(topology: Topology, edges: Point, angles: Scalar, normals: Patch, directions: Point) -> Plane:
    """The vertex gradient (`Plane+[V]`) of the developability energy at fixed least-spread `directions` (`Point-[V]`) sums each triangle's normal-turning and corner-angle variations."""
    double_areas = (edges[0] & edges[1]).norm()                                 # Scalar+[F]
    corner_directions = topology.incidence[2, 0] * directions                   # [3] Point-[F]
    # Project each corner's minimizing direction onto the triangle's unit normal patch.
    normal_projections = normals & corner_directions                            # [3] Scalar-[F]
    weighted_directions = (corner_directions * (normal_projections * angles)).sum(axis=0)  # Bivector+[F]
    tangent_directions = 2 * (weighted_directions - normals.dual() * (weighted_directions & normals))  # Bivector+[F]
    # Join the tangential direction rate with the opposite edge over twice the triangle area.
    normal_variation = (tangent_directions & edges) / double_areas              # [3] Plane+[F]
    # Moving an edge's endpoints oppositely along its in-plane normal opens one corner angle and closes the other.
    tails, heads = topology.corners[2, 1]                                       # [3] Scalar+[F <- F], [3] Scalar+[F <- F]
    angle_variation = (edges | normals) / edges.scalar_norm_squared() * (normal_projections[heads].squared() - normal_projections[tails].squared())  # [3] Plane+[F]
    return (topology.incidence[2, 0].adjugate() * normal_variation + (topology.oriented[2, 1] * topology.d[0]).adjugate() * angle_variation).sum(axis=0)  # Plane+[V]


initial_gradient = energy_gradient(topology, edges, angles, normals, directions[:, 0].field())  # Plane+[V]
gradient_norms = initial_gradient.norm().batch()                                # [V] Scalar+
print("largest vertex gradient norm:", gradient_norms[gradient_norms.argmax()].to_array())

## Stiffness-scaled developability flow

Moving a corner turns its triangle's normal in inverse proportion to the corner's altitude above the opposite edge, so dividing each vertex gradient by the local normal-turning curvature keeps steps stable across tiny ear triangles and broad flank triangles alike. Compiling each block of `n_stride` steps with `jax.lax.scan` and rendering the flat-shaded frames takes around ten seconds.

In [ ]:
step_scale = 0.3
n_steps = 600
n_stride = 5
n_states = n_steps // n_stride + 1
frame_duration_ms = 40

In [ ]:
def vertex_stiffness(topology: Topology, edges: Point, angles: Scalar) -> Scalar:
    """Diagonal normal-turning curvature (`Scalar+[V]`) sums each incident corner's angle divided by its squared altitude above the opposite edge."""
    double_areas = (edges[0] & edges[1]).norm()                                 # Scalar+[F]
    return (topology.incidence[2, 0].adjugate() * (angles * edges.scalar_norm_squared() / double_areas.squared())).sum(axis=0)  # Scalar+[V]


def developability_flow(topology: Topology, vertices: Point, step_scale: float, n_states: int, n_stride: int) -> Iterator[tuple[Point, Scalar]]:
    """Yields `n_states` snapshots of vertex positions (`Point-[V]`) and minimal spread eigenvalues (`Scalar+[V]`) every `n_stride` stiffness-scaled gradient steps."""
    def flow_step(current_vertices: Point, _: None) -> tuple[Point, Scalar]:
        edges = topology.boundary[2, 1] * (topology.d[0] * current_vertices)    # [3] Point-[F]
        angles = corner_angles(topology, edges)                                 # [3] Scalar+[F]
        normals, spreads = normal_spread(topology, edges, angles)               # Patch-[F], (Point- <- Point-)[V]
        eigenvalues, directions = spreads.batch().eigh()                        # [V, 3] Scalar+, [V, 3] Point-
        # Convert the stiffness-scaled gradient covector into an intrinsic vertex step via the metric dual.
        step = (energy_gradient(topology, edges, angles, normals, directions[:, 0].field()) / vertex_stiffness(topology, edges, angles)).dual()  # Point-[V]
        return current_vertices - step_scale * step, eigenvalues[:, 0]

    stride_scan = jax.jit(lambda current_vertices: jax.lax.scan(flow_step, current_vertices, length=n_stride))
    vertices = vertices.materialize(JaxContext(ga))                             # Point-[V]
    for _ in range(n_states):
        next_vertices, step_values = stride_scan(vertices)                      # Point-[V], [n_stride] Scalar+
        yield vertices.materialize(mv.context), step_values[0].materialize(mv.context).field()
        vertices = next_vertices


states = show_states(
    developability_flow(topology, vertices, step_scale, n_states, n_stride),
    render.flow_animation, topology,
    name="developability_flow", duration_ms=frame_duration_ms, n_frames=n_states,
);

Small scan bumps disappear first, after which the rounded flanks, back, and head flatten into broad developable panels joined along crisp creases where only two normal directions meet.

In [ ]:
energies = stack([step_eigenvalues.batch().sum(axis=0) for _, step_eigenvalues in states])  # [n_states] Scalar+
print("developability energy before and after:", energies[0].to_array(), energies[-1].to_array())

In [ ]:
# checks
assert ((normals.norm() - 1).abs() < 1e-6).all()
assert (eigenvalues >= -1e-12).all()
assert initial_gradient.batch().sum(axis=0).norm() < 1e-10
assert (energies[1:] < energies[:-1]).all()
assert energies[-1] < 0.1 * energies[0]